# UdaPlay AI Research Agent

## Part 1 - Offline RAG Pipeline

This notebook implements the first stage of the UdaPlay project: a local Retrieval-Augmented Generation (RAG) pipeline for video game data.

The pipeline will:

- Load the provided game JSON files
- Store them in a ChromaDB vector database
- Generate embeddings for semantic search
- Retrieve relevant game information from the local knowledge base
- Create reusable retrieval components for the agent workflow

The source data is stored in the `games` folder. Each JSON file represents one game and will be added as a document in the Chroma collection.

Example game record:

```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [12]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [13]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [15]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [21]:
import chromadb

chroma_client = chromadb.PersistentClient(path="chromadb")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Collection

In [22]:
# TODO: Pick one embedding function
from chromadb.utils import embedding_functions
import os

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-small"
)

In [23]:
# TODO: Create a collection
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Add documents

In [24]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)

    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Text used for semantic search
    content = (
        f"Game: {game['Name']}. "
        f"Platform: {game['Platform']}. "
        f"Genre: {game['Genre']}. "
        f"Publisher: {game['Publisher']}. "
        f"Year of release: {game['YearOfRelease']}. "
        f"Description: {game['Description']}"
    )

    # Use the JSON filename as the unique document ID
    doc_id = os.path.splitext(file_name)[0]

    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )